# 1、使用Function_Call方式

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、提示词模版（ChatPromptTemplate为例）
prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个乐于助人的ai助手，根据用户的提问，必要时调用search工具，使用互联网检索数据"),
    ("human","{input}"),
    ("system","{agent_scratchpad}") # agent_scratchpad必须声明
])
# 5、获取Agent实例：create_tool_calling_agent()
agent = create_tool_calling_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...

Invoking: `search` with `西安天气预报`


[{'title': '西安-天气预报 - 中央气象台', 'url': 'https://www.nmc.cn/publish/forecast/ASN/xian.html', 'content': '+ 土壤水分监测\n  + 农业干旱综合监测\n  + 关键农时农事\n  + 农业气象周报\n  + 农业气象月报\n  + 生态气象监测评估\n  + 农业气象专报\n  + 作物发育期监测\n  + 农业气象灾害风险预警\n  + 国外农业气象月报\n 数值预报\n\n  + CMA全球天气模式\n  + CMA全球集合模式\n  + CMA区域模式\n  + CMA区域集合模式\n  + CMA台风模式\n  + 海浪模式\n\n1. 当前位置：首页\n2. 陕西省\n3. 西安天气预报\n\n省份：城市：\n\n日出\n\n西安\n\n日落\n\n降水量\n\n相对湿度\n\n体感温度\n\n雷达图\n\n24小时预报7天预报10天预报11-30天预报\n\n发布时间：03-23 12:00\n\n03/23   \n周一\n\n小雨\n\n无持续风向\n\n微风\n\n12℃\n\n7℃\n\n小雨\n\n无持续风向\n\n微风\n\n03/24   \n周二\n\n多云\n\n无持续风向\n\n微风\n\n18℃\n\n6℃\n\n多云\n\n无持续风向\n\n微风\n\n03/25   \n周三\n\n阴\n\n无持续风向\n\n微风\n\n20℃\n\n7℃\n\n阴\n\n无持续风向\n\n微风\n\n03/26   \n周四\n\n阴\n\n无持续风向\n\n微风\n\n17℃\n\n8℃\n\n多云\n\n无持续风向\n\n微风\n\n03/27   \n周五\n\n晴\n\n无持续风向\n\n微风\n\n21℃\n\n9℃\n\n阴\n\n无持续风向\n\n微风\n\n03/28   \n周六\n\n多云\n\n无持续风向\n\n微风\n\n21℃\n\n8℃\n\n阴\n\n无持续风向\n\n微风\n\n03/29   \n周日\n\n阴\n\n无持续风向\n\n微风\n\n16℃\n\n

# 2、使用ReAct方式

示例1：使用PromptTemplate实现

In [4]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor, create_react_agent
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、提示词模版（PromptTemplate为例）
template = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
"""
prompt_template = PromptTemplate.from_template(
    template = template,
)
# 5、获取Agent实例：create_react_agent()
agent = create_react_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...
Action: search  
Action Input: '西安天气情况 2023年10月25日'  [{'title': '西安市在十月2026 的天氣：氣溫與氣候 - Climate-Data.org', 'url': 'https://zh.climate-data.org/%E4%BA%9E%E6%B4%B2/%E4%B8%AD%E5%9B%BD/%E9%99%95%E8%A5%BF/%E8%A5%BF%E5%AE%89%E5%B8%82-2613/t/%E5%8D%81%E6%9C%88-10/', 'content': 'Partner-Links\n预订您的酒店：\n西安市\n预订您的度假公寓：\n西安市\n飞往以下城市的特价航班：\n西安市\n\n## 十月 西安市 的气温\n\n氣候圖 // 每月天氣, 西安市\n\n十月 在 西安市 中最熱的一天是 02.10，當天最高溫為 21.9°C，而最冷的一天是 31.10，最低溫降至 6.6°C。 月初的日最高平均溫度為 20.8°C，最低平均溫度為 12.2°C。 月中時，日最高平均溫度約為 19.1°C，最低平均溫為 10.2°C。 到月底，平均最高溫約為 17.3°C，最低溫接近 7.8°C。 最熱的連續 10 天期間是 1-10，日最高溫約為 16.3°C。 最冷的連續 10 天區間是 21-31，平均最低溫為 12.4°C。\n\n十月 在 西安市 的日照時間平均介於每天 7 小時至 9 小時之間。最明亮的一天 07.10 大約有 9 小時，而日照最少的 24.10 則只有 7 小時。月初三分之一期間的日照平均為 8 小時；第二個三分之一為 8 小時；月底三分之一為 8 小時。最晴朗的 10 天階段是從 1-10，每天約有 8 小時日照。最陰天的階段是 21-31，只有 8 小時。\n\n在 十月 ，西安市 的降水量介於每日總量 2.7 毫米/英吋與 14.3 毫米/英吋之間波動。最濕潤的一天是 30.10，降水量為 14.3 毫米/英吋，最乾燥的則是 01.10，僅有 2.7 毫米/英吋。前十天的平均降雨量為 5.6 毫米/英吋，當中有 9 天降雨量超過 1 毫米。第 11 至 20 天期間的平均降雨量為 7

In [7]:
from langchain_classic import hub
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor, create_react_agent
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、使用LangChain Hub中的官方ReAct提示模板
prompt_template = hub.pull("hwchase17/react")

# 5、获取Agent实例：create_react_agent()
agent = create_react_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...
我需要查找今天西安的天气情况。  
Action: search  
Action Input: '西安天气 2023年10月5日'  [{'title': '春季多沙尘天气霜降期间“暖如夏” 西安发布2023年重大天气气候事件', 'url': 'http://m.cnwest.com/sxxw/a/2024/01/05/22219229.html', 'content': '汛期暴雨频发极端性强\n\n2023年汛期西安市强对流天气频发，具有局地性强、致灾性强等特点。其中7月12-13日，西安市出现区域性暴雨天气，极端性强，日最大降水量89.6毫米（蓝田12日20时-13日20时），蓝田12日08时-13日08时降水量169.0毫米，突破历史日最大降水纪录。13日区域自动站最大日降水量为鄠邑太平林木管护站161.5毫米。8月11日，西安市出现强对流天气，长安、西咸新区、临潼局地暴雨，蓝田局地大暴雨，全市共95站出现降水，其中10站降水量大于50毫米，长安区滦镇街道喂子坪村鸡窝子组突发山洪泥石流灾害，道路、桥梁、供电、通信等基础设施损毁严重。\n\n华西秋雨期为1961年以来最长\n\n2023年西安市华西秋雨于9月18日开始，较常年偏晚3天，结束时间为10月7日，秋雨期长度为20天，较常年偏少5天，雨量达158.0毫米，较常年偏多1.6 倍，综合强度等级为正常等级。9月19日全市出现37站暴雨。9月18日-10月6日的持续阴雨天气，持续时间长度为1961年以来9-10月最长，全市平均降水量158.0毫米，是常年同期的1.6倍，为2006年以来同期第二多，为一次重度区域性连阴雨过程。\n\n霜降期间“暖如夏”\n\n2023年霜降（10月24日）节气期间出现异常回暖天气，气温不降反升。10月24日-11月2日，全市平均气温15.4℃，较常年同期偏高4.0℃，为1961年以来同期最高。期间各区县最高气温均达26℃以上，高陵、市区日最高气温突破11月历史极值。\n\n11月现罕见“雷打雪”天气 [...] 11月现罕见“雷打雪”天气\n\n2023年11月11日清晨伴随阵阵雷声，西安市降下冬季第一场雪，初雪时间较常年偏早，历史最早为1981年11月5日。根据雷暴观测数据，历史上西安最晚雷暴

示例2：使用ChatPromptTemplate实现

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import AgentType, initialize_agent, create_tool_calling_agent, AgentExecutor, create_react_agent
from langchain_core.tools import StructuredTool
from langchain_classic.tools import Tool
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

# 读取配置文件
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取Tavily搜索工具实例
search = TavilySearchResults(max_results = 3)
# 2、获取对应工具实例

search_tool = Tool(
    func = search.run,
    name = "search",
    description = "用于检索互联网上的信息",
)
# 3、获取大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)
# 4、提示词模版（ChatPromptTemplate为例）
system_template = """
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system",system_template),
    ("human","{input}"),
    ("system","{agent_scratchpad}") # agent_scratchpad必须声明
])
# 5、获取Agent实例：create_react_agent()
agent = create_react_agent(
    llm = llm,
    prompt = prompt_template,
    tools = [search_tool],
)
# 6、获取AgentExecutor实例
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True,
)
# 7、通过AgentExecutor实例调用invoke方法，得到响应
response = agent_executor.invoke({"input":"查询今天的西安天气情况"})
# 8、处理响应
print(response)



> Entering new AgentExecutor chain...
Question: 查询今天的西安天气情况
Thought: 我需要查找今天西安的天气情况。
Action: search
Action Input: "西安天气预报 2023年10月25日"[{'title': '西安市10月份气温查询- 西安历史天气 - 天气二四网', 'url': 'https://www.tianqi24.com/xian/history10.html', 'content': '# 西安市10月份气温西安历史天气\n\n## 西安10月份气温怎样？\n\n西安市十月份气温，根据2025年、2024年、2023年中每年的10月的历史天气数据统计分析，共有：\n\n平均高温：21℃， 最高温度(2024年10月15日)： 29℃，\n\n平均低温：11℃； 最低温度(2025年10月21日)： 3℃；\n\n平均空气综合质量AQI值是： 42 (优)\n\n空气最好：10 优， 空气最差：85 良；\n\n西安近3年的每年10月份：月均降雨（8）天, 月均降雨量（188.87）毫米！\n\n  \n\n### 西安2025年、2024年、2023年10月天气分布图\n\n  \n  \n\n### 西安10月温度走势图\n\n### 西安市10月份气温： 2023年 ~ 2025年\n\n 雨12天  \n  晴8天  \n  阴6天  \n  多云5天\n\n  多云10天  \n  晴9天  \n  雨7天  \n  阴5天\n\n  晴11天  \n  阴7天  \n  多云7天  \n  雨5天  \n  沙雾霾1天\n 11℃\n\n  10℃\n 月降雨量\n\n  479.3 mm\n\n  0.3 mm\n\n  87 mm\n 日均PM25\n\n  25 优\n\n  40 优\n\n  29 优\n 日均最高风速\n\n  7.8 Km/h\n\n  8.3 Km/h\n\n  5.6 Km/h\n 该月最高温度\n\n  25℃  \n  10月08日\n\n  29℃  \n  10月15日\n\n  26℃  \n  10月15日\n 该月最低温度\n\n  7℃  \n  10月21日\n\n  3℃  \n  10月23日\n\n 

传统方式相较通用方式来说不提供提示词模版

对于通用方式来说：

①FUNCTION_CALL模式：创建agent时，推荐使用ChatPromptTemplate

②ReAct模式：创建agent时，ChatPromptTemplate和PromptTemplate均可，更推荐第二个